# 🧐 Train on Test?!

Hi Kagglers!  
Here I want to show how it’s possible to get a strong score with a small model by using information mined from the test data itself during training.


## 📌 The Idea

In `test.csv` there are columns `positive_example_1/2` and `negative_example_1/2`.  
For each rule in the test set, these contain *labeled* example comments that either violate or do not violate that rule.

The experiment here:
- Take these examples from the test set,
- Add them to the training data,
- Fine‑tune a small model with LoRA during the submission run.

⚡ Even with a small model, this approach can noticeably boost the metric — without using huge LLMs.


## 🤔 Why It’s Unclear

The competition rules say:

> [4. COMPETITION ENTRY. b.](https://www.kaggle.com/competitions/jigsaw-agile-community-rules/rules#4.-competition-entry) "Submissions may not use or incorporate information from hand labeling or human prediction of the validation dataset or test data records."

I couldn’t find anything in the rules that exactly covers our case — when we mine information from the test data itself and use it for training.

From one side — we are not labeling the `body` field manually; we are using provided features and not extracting the target for `body`.  
From the other — we are still using part of the test set as training data.


## 🗣 Purpose of This Notebook

- Highlight a possible grey area in the current rule wording.
- Show the impact this approach can have, even with a small model.
- Encourage discussion among participants and possibly the organizers.
- Gather opinions on whether this is acceptable in the context of this competition.


## 💬 Call for Discussion

What do you think — is it okay to do this?


# Code

## Cell 1: `constants.py` — Model & Prompt Configuration

Writes `constants.py` to disk:
- `BASE_MODEL_PATH`: Path to the quantized Qwen2.5-7B-Instruct (GPTQ INT4) base model
- `LORA_PATH`: Directory where the fine-tuned LoRA adapter will be saved
- `DATA_PATH`: Location of the Jigsaw dataset
- `POSITIVE_ANSWER` / `NEGATIVE_ANSWER`: The exact single tokens the model must output (`"Yes"` / `"No"`) — kept to single tokens so `MultipleChoiceLogitsProcessor` can constrain decoding to just these two choices
- `BASE_PROMPT` / `COMPLETE_PHRASE`: System-level instruction prepended to every example; `COMPLETE_PHRASE` marks where the model's answer begins

In [1]:
%%writefile constants.py
BASE_MODEL_PATH = "/kaggle/input/qwen2.5/transformers/7b-instruct-gptq-int4/1"
#BASE_MODEL_PATH = "/kaggle/input/qwen-3/transformers/1.7b-gptq-int8/1"
LORA_PATH = "output/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = f"Reddit moderation: Does the comment violate the rule? Answer '{POSITIVE_ANSWER}' or '{NEGATIVE_ANSWER}' only."

Writing constants.py


## Cell 2: Module Cache Clearing (Disabled)

This cell is wrapped in a triple-quoted string to disable it. It clears Python's module cache for `utils` and invalidates `importlib` caches — useful during iterative development when `utils.py` is rewritten and re-imported in the same kernel session. Safe to ignore in a clean run.

In [2]:
'''
import sys, importlib, shutil, os

# Remove old references
for m in list(sys.modules):
    if m.startswith("utils"):
        del sys.modules[m]

# Clear caches
importlib.invalidate_caches()
shutil.rmtree("__pycache__", ignore_errors=True)
'''

'\nimport sys, importlib, shutil, os\n\n# Remove old references\nfor m in list(sys.modules):\n    if m.startswith("utils"):\n        del sys.modules[m]\n\n# Clear caches\nimportlib.invalidate_caches()\nshutil.rmtree("__pycache__", ignore_errors=True)\n'

## Cell 3: `utils.py` — Prompt Builder & Dataset Utilities

Writes `utils.py` to disk with three functions:
- **`build_prompt(row)`**: Formats each example as a numbered list (`Subreddit / Rule / Comment`) followed by `"Answer:"`, matching the instruction-following format expected by Qwen2.5-Instruct
- **`get_dataframe_to_train(data_path)`**: Builds the full labeled corpus (train split + positive/negative examples from the test set), deduplicates, then **samples 15%** at random to keep fine-tuning within Kaggle's runtime limit
- **`build_dataset(dataframe)`**: Applies `build_prompt`, maps `rule_violation` to `"Yes"` / `"No"` as the `completion` target, and returns a HuggingFace `Dataset` ready for `SFTTrainer`

In [3]:
%%writefile utils.py
import pandas as pd
from datasets import Dataset
from constants import POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT


def build_prompt(row):
    """Variation 4: Numbered format"""
    return f"""
Reddit moderation: Does the comment violate the rule? Answer '{POSITIVE_ANSWER}' or '{NEGATIVE_ANSWER}' only.

1. Subreddit: r/{row["subreddit"]}
2. Rule: {row["rule"]}
3. Comment: {row["body"]}
---
Answer:"""


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(ignore_index=True)
    
    # use a fraction for finetuning 
    dataframe=dataframe.sample(frac=0.15, random_state=42).reset_index(drop=True)
    
    return dataframe


def build_dataset(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]
    if "rule_violation" in dataframe:
        dataframe["completion"] = dataframe["rule_violation"].map(
            {
                1: POSITIVE_ANSWER,
                0: NEGATIVE_ANSWER,
            }
        )
        columns.append("completion")

    dataframe = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe)
    return dataset

Writing utils.py


## Cell 4: `train.py` — LoRA Fine-tuning with DeepSpeed

Writes the training script to disk. The `main()` function:
1. Samples 15% of the labeled corpus and builds the SFT dataset
2. Configures **LoRA** (rank=16, alpha=32, dropout=0.1) targeting all attention and MLP projection layers (`q/k/v/o/gate/up/down_proj`)
3. Trains for 1 epoch using `paged_adamw_8bit` (lr=5e-4, cosine schedule, 5% warmup) with gradient checkpointing; `completion_only_loss=True` computes loss only on the `"Yes"/"No"` answer token, not the full prompt
4. Saves the LoRA adapter weights to `LORA_PATH`

Designed to run under `accelerate` + DeepSpeed ZeRO-2 across 2 GPUs (see `accelerate_config.yaml`).

In [4]:
%%writefile train.py
import pandas as pd

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, BASE_MODEL_PATH, LORA_PATH


def main():
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)

    print(BASE_MODEL_PATH)
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    
    training_args = SFTConfig(
        num_train_epochs=1,
        
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        
        optim="paged_adamw_8bit",
        learning_rate=5e-4,
        weight_decay=0.01,
        max_grad_norm=1.0,
        
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        
        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),
        dataloader_pin_memory=True,
        
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    
        save_strategy="no",
        report_to="none",
    
        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
    )
    
    trainer = SFTTrainer(
        BASE_MODEL_PATH,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )
    
    trainer.train()
    trainer.save_model(LORA_PATH)


if __name__ == "__main__":
    main()

Writing train.py


## Cell 5: `inference.py` — Constrained vLLM Inference

Writes the inference script to disk. The `main()` function:
1. Loads the base model via **vLLM** with GPTQ quantization and tensor parallelism across all available GPUs; `enable_lora=True` allows hot-swapping LoRA adapters at inference time
2. Attaches the fine-tuned adapter via `LoRARequest` without reloading the base model weights
3. Uses `MultipleChoiceLogitsProcessor` to **hard-constrain** generation to `"Yes"` or `"No"` only — all other tokens are masked, making the output deterministic and removing any need for threshold tuning
4. Extracts the **log-probability of `"Yes"`** (the violation class) as a continuous ranking score
5. Saves `submission.csv` with `row_id` and `rule_violation` (log-prob of `"Yes"`; more negative = less likely to be a violation)

In [5]:
%%writefile inference.py
import os
os.environ["VLLM_USE_V1"] = "0"

import vllm
import torch
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from datasets import Dataset
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import BASE_MODEL_PATH, LORA_PATH, DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER


def main():
    llm = vllm.LLM(
        BASE_MODEL_PATH,
        quantization="gptq",
        tensor_parallel_size=torch.cuda.device_count(),
        gpu_memory_utilization=0.9,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=4096,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )
    
    
    tokenizer = llm.get_tokenizer()
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])
    
    
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataset = build_dataset(test_dataframe)
    
    texts = test_dataset["prompt"]
    outputs = llm.generate(
        texts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )
    
    log_probs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    
    predictions = pd.DataFrame(log_probs)[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    predictions["row_id"] = test_dataframe["row_id"]
    submission = predictions[["row_id", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    submission.to_csv("submission.csv", index=False)

if __name__ == "__main__":
    main()

Writing inference.py


## Cell 6: `accelerate_config.yaml` — DeepSpeed ZeRO-2 Configuration

Writes the Accelerate/DeepSpeed config to disk:
- **ZeRO Stage 2**: partitions optimizer states and gradients across 2 GPUs, halving per-GPU memory without the communication overhead of Stage 3 (which also partitions parameters)
- **FP16** with dynamic loss scaling (`initial_scale_power=16`)
- `train_micro_batch_size_per_gpu=4` × `gradient_accumulation_steps=8` × 2 GPUs → effective batch size of 64
- `INDUCTOR` dynamo backend for kernel fusion via `torch.compile`
- Optimizer and parameters stay on GPU (`offload_*_device: none`) to maximize throughput

In [6]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 8
  gradient_clipping: 1.0
  train_batch_size: 64
  train_micro_batch_size_per_gpu: 4
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

Writing accelerate_config.yaml


## Cell 7: Run LoRA Fine-tuning

Launches `train.py` under `accelerate` with the DeepSpeed ZeRO-2 config across 2 GPUs. Trains for 1 epoch on ~297 samples (15% of the corpus), completing in ~10 minutes (5 gradient steps with accumulation). The LoRA adapter is saved to `output/`.

In [7]:
!accelerate launch --config_file accelerate_config.yaml train.py

[2025-09-05 17:33:45,630] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-09-05 17:33:48,980] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
2025-09-05 17:33:51.608244: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757093631.949624     187 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757093632.046546     187 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0905 17:34:06.540000 187 torch/distributed/run.py:766] 
W0905 17:34:06.540000 187 torch/distributed/run.py:766] *****************************************
W0905 17:34:06.540000 187 torc

## Cell 8: Run Inference

Runs `inference.py` to load the fine-tuned LoRA adapter on top of the quantized base model via vLLM and generate constrained `"Yes"`/`"No"` predictions for the 10 test comments. The log-probability of `"Yes"` is saved as the continuous `rule_violation` score in `submission.csv`.

In [8]:
!python inference.py

2025-09-05 17:47:36.565210: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757094456.587910     401 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757094456.594858     401 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 09-05 17:47:41 [__init__.py:235] Automatically detected platform cuda.
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 09-05 17:47:58 [config.py:1604] Using max model len 4096
WARNING 09-05 17:47:59 [config.py:1084] gptq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 09-05 17:48:00 [cuda.py:103] To see benefits of async output processing, enable CUDA graph. Since, enfor

## Cell 9: Preview Submission

Prints the first few rows of `submission.csv` to verify the output format. The `rule_violation` values are log-probabilities of `"Yes"` — all negative since log-probs are ≤ 0, with values closer to 0 indicating higher confidence in a rule violation.

In [9]:
!head submission.csv

row_id,rule_violation
2029,-0.5828105211257935
2030,-0.20718736946582794
2031,-0.16730396449565887
2032,-0.16491271555423737
2033,-0.09872718155384064
2034,-0.2450537085533142
2035,-0.11967968195676804
2036,-0.3672420084476471
2037,-0.14903587102890015
